In [1]:
from scipy.io import loadmat
import numpy as np
import torch

file_path = "data/BCICIV_calib_ds1a_1000Hz.mat"
device = torch.device('cuda') if torch.cuda.is_available() else torch.device("cpu")

mat = loadmat(file_path)

mat.keys()

dict_keys(['__header__', '__version__', '__globals__', 'cnt', 'mrk', 'nfo'])

In [2]:
freq = mat["nfo"]["fs"][0][0][0][0]
cls_names = mat["nfo"]["classes"][0][0][0]

image_display_time_s = 4
blank_display_time_s = 2
fixation_only_display_time_s = 2

eeg_signal = mat["cnt"]
n_channels = eeg_signal.shape[1]
n_samples =  eeg_signal.shape[0]
n_classes = 2
batch_size = 16

target_classes = mat["mrk"]["y"][0][0][0]
image_show_index = mat["mrk"]["pos"][0][0][0]

cls_names

array([array(['left'], dtype='<U4'), array(['foot'], dtype='<U4')],
      dtype=object)

In [3]:
trial_len = int(freq * image_display_time_s)  # 4 seconds

X_trials = []
y_trials = []
for pos, cls in zip(image_show_index, target_classes):
    start = pos
    end = pos + trial_len
    if end <= n_samples:
        X_trials.append(eeg_signal[start:end])  # (T, C)
        y_trials.append(int((cls+1)/2))

X_trials = torch.tensor(np.stack(X_trials)).float().permute(0, 2, 1)  # (N_trials, C, T)
y_trials = torch.tensor(y_trials).long()

mean = X_trials.mean(dim=(0, 1), keepdim=True)
std = X_trials.std(dim=(0, 1), keepdim=True)
X_trials = (X_trials - mean) / (std + 1e-6)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_trials, y_trials, test_size=0.2)

In [5]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

class BasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock1D, self).__init__()
        # 1st Conv with Stride
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        
        # 2nd Conv
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)

        # Shortcut connection handling
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ResNet1D(nn.Module):
    def __init__(self, block, num_blocks, input_channels=59, num_classes=2):
        super(ResNet1D, self).__init__()
        self.in_planes = 64

        # Initial Convolution (Stem)
        # Using kernel_size=7 and stride=2 to downsample initial high-res signal quickly
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        # ResNet Layers
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        # Classification Head
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        # x shape: [Batch, Channels, Length]
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

def ResNet18_1D(input_channels, num_classes):
    # ResNet18 config: [2, 2, 2, 2] blocks
    return ResNet1D(BasicBlock1D, [2, 2, 2, 2], input_channels=input_channels, num_classes=num_classes)

In [7]:
LEARNING_RATE = 0.001
EPOCHS = 10

model = ResNet18_1D(input_channels=59, num_classes=n_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss/len(train_dataloader):.4f}")

Epoch [1/10] Loss: 1.3140
Epoch [2/10] Loss: 0.7299
Epoch [3/10] Loss: 0.6952
Epoch [4/10] Loss: 0.6966
Epoch [5/10] Loss: 0.5997
Epoch [6/10] Loss: 0.5785
Epoch [7/10] Loss: 0.4750
Epoch [8/10] Loss: 0.5488
Epoch [9/10] Loss: 0.5380
Epoch [10/10] Loss: 0.4590


In [9]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\nFinal Train Accuracy: {100 * correct / total:.2f}%")


Final Train Accuracy: 80.00%


In [10]:
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\nFinal Test Accuracy: {100 * correct / total:.2f}%")


Final Test Accuracy: 57.50%
